In [1]:
import math
import random
import numpy as np
import sympy as sp
import jpegio as jio
from pathlib import Path
from scipy.fftpack import dct
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

random.seed(10)

BASE_DIR = Path.cwd()

SOURCE_IMAGE_DIR = BASE_DIR / "test_8"
BOSSBASE_DIR = BASE_DIR / "test_200"
GRAY_IMAGE_DIR = BASE_DIR / "test_50"
STEGO_IMAGE_DIR = BASE_DIR / "Gray_image_stego"


### Helper Functions

In [2]:
def mse(image_a, image_b):
    return np.square(
        np.subtract(image_a.astype(float), image_b.astype(float))
    ).mean()


def psnr(image_a, image_b):
    return 10 * np.log10(255 ** 2 / mse(image_a, image_b))

def found_max(image_name, quality_factor, zigzag_mode, embedding_mode, file_path):
    """Count the embeddable coefficients under the current settings."""
    jpeg = jio.read(
        str(file_path / f"{image_name}_{quality_factor}.jpg")
    )

    # Quantized DCT coefficients of the Y channel
    quantized_dct = jpeg.coef_arrays[0]
    image_height, image_width = quantized_dct.shape
    block_rows = image_height // 8
    block_cols = image_width // 8

    # Read the JPEG quantization table
    quant_table = jpeg.quant_tables[0]

    if zigzag_mode == 1:
        coefficient_order = np.append(
            0,
            np.argsort(quant_table.flatten("C")[1:], kind="mergesort") + 1,
        )
    else:
        coefficient_order = np.array([
            0, 1, 8, 16, 9, 2, 3, 10, 17, 24, 32, 25, 18, 11, 4, 5,
            12, 19, 26, 33, 40, 48, 41, 34, 27, 20, 13, 6, 7, 14, 21,
            28, 35, 42, 49, 56, 57, 50, 43, 36, 29, 22, 15, 23, 30,
            37, 44, 51, 58, 59, 52, 45, 38, 31, 39, 46, 53, 60, 61,
            54, 47, 55, 62, 63,
        ])

    # Convert quantized DCT coefficients to shape (block_count, 64)
    dct_blocks = quantized_dct.reshape(
        block_rows, 8, block_cols, 8
    ).swapaxes(1, 2).reshape(block_rows * block_cols, 64).astype(int)

    if embedding_mode == 0:
        block_sort_key = np.count_nonzero(dct_blocks, axis=1)
        sorted_index = np.argsort(block_sort_key)
        sorted_blocks = dct_blocks[sorted_index, :]

        embeddable_coefficients = np.array([], dtype=int)
        for order_index in range(1, len(coefficient_order)):
            coefficients = sorted_blocks[:, coefficient_order[order_index]]
            embeddable_coefficients = np.concatenate(
                (embeddable_coefficients, coefficients[coefficients != 0])
            )
    else:
        last_nonzero_positions = np.zeros(block_rows * block_cols, dtype=int)

        for block_index in range(block_rows * block_cols):
            ordered_coefficients = dct_blocks[block_index, coefficient_order]
            nonzero_positions = np.where(ordered_coefficients != 0)[0]
            if len(nonzero_positions) > 0:
                last_nonzero_positions[block_index] = nonzero_positions[-1]

        sorted_index = np.argsort(last_nonzero_positions)
        sorted_blocks = dct_blocks[sorted_index, :]
        sorted_last_nonzero = last_nonzero_positions[sorted_index]

        embeddable_coefficients = np.array([], dtype=int)
        for order_index in range(1, len(coefficient_order)):
            coefficients = sorted_blocks[:, coefficient_order[order_index]]
            embeddable_coefficients = np.concatenate(
                (
                    embeddable_coefficients,
                    coefficients[sorted_last_nonzero > order_index],
                )
            )

    return len(embeddable_coefficients)

def bitsum_emd(group_flags, group_size):
    """Calculate the theoretical number of bits carried by the current EMD groups."""
    return len(group_flags) * math.log2(group_size * 2 + 1)


def ftake(coefficients, group_size):
    modulus = 2 * group_size + 1
    weighted_sum = 0

    for index, coefficient in enumerate(coefficients, 1):
        weighted_sum += coefficient * index

    return weighted_sum % modulus


def mod(target_symbol, coefficients, group_size):
    difference = target_symbol - ftake(coefficients, group_size)
    modulus = 2 * group_size + 1

    if difference >= 0:
        return difference

    return modulus - abs(difference)


def emd_en(coefficients, target_symbol, group_size):
    modified_coefficients = coefficients
    modulus = 2 * group_size + 1

    if ftake(coefficients, group_size) == target_symbol:
        return modified_coefficients

    while ftake(coefficients, group_size) != target_symbol:
        offset = mod(target_symbol, modified_coefficients, group_size)

        if offset <= group_size:
            modified_coefficients[offset - 1] += 1
        else:
            modified_coefficients[modulus - offset - 1] -= 1

        if ftake(coefficients, group_size) == target_symbol:
            return modified_coefficients


def diff_group_2_200(payload_bits, image_name, quality_factor, group_size, file_path):
    """Perform EMD embedding with an arbitrary group size n (zero coefficients may also be candidate positions)."""
    random.seed(10)
    embedded_symbols = ""

    jpeg = jio.read(
        str(file_path / f"{image_name}_{quality_factor}.jpg")
    )

    # Quantized DCT coefficients of the Y channel
    quantized_dct = jpeg.coef_arrays[0]
    image_height, image_width = quantized_dct.shape
    block_rows = image_height // 8
    block_cols = image_width // 8

    quant_table = jpeg.quant_tables[0]
    coefficient_order = np.append(
        0,
        np.argsort(quant_table.flatten("C")[1:], kind="mergesort") + 1,
    )

    dct_blocks = quantized_dct.reshape(
        block_rows, 8, block_cols, 8
    ).swapaxes(1, 2).reshape(block_rows * block_cols, 64).astype(int)

    last_nonzero_positions = np.zeros(block_rows * block_cols, dtype=int)
    for block_index in range(block_rows * block_cols):
        ordered_coefficients = dct_blocks[block_index, coefficient_order]
        nonzero_positions = np.where(ordered_coefficients != 0)[0]

        if len(nonzero_positions) > 0:
            last_nonzero_positions[block_index] = nonzero_positions[-1]

    sorted_index = np.argsort(last_nonzero_positions)
    sorted_blocks = dct_blocks[sorted_index, :]
    sorted_last_nonzero = last_nonzero_positions[sorted_index]
    modified_sorted_blocks = sorted_blocks.copy()

    embeddable_coefficients = np.array([], dtype=int)
    for order_index in range(1, len(coefficient_order)):
        coefficients = sorted_blocks[:, coefficient_order[order_index]]
        embeddable_coefficients = np.concatenate(
            (
                embeddable_coefficients,
                coefficients[sorted_last_nonzero > order_index],
            )
        )

    coefficient_groups = []
    for start_index in range(0, len(embeddable_coefficients), group_size):
        coefficient_groups.append(
            embeddable_coefficients[start_index:start_index + group_size]
        )

    embedded_group_flags = ""
    for group_index in range(len(coefficient_groups)):
        if (
            len(coefficient_groups[group_index]) == group_size
            and bitsum_emd(embedded_group_flags, group_size) <= payload_bits
        ):
            target_symbol = random.randint(0, group_size * 2)
            embedded_symbols += str(target_symbol)
            embedded_group_flags += "0"
            coefficient_groups[group_index] = emd_en(
                coefficient_groups[group_index],
                target_symbol,
                group_size,
            )

    flattened_coefficients = []
    for group in coefficient_groups:
        flattened_coefficients.extend(group)

    coefficient_offset = 0
    for order_index in range(1, len(coefficient_order)):
        mask = sorted_last_nonzero > order_index
        coefficient_count = len(
            sorted_blocks[:, coefficient_order[order_index]][mask]
        )
        modified_sorted_blocks[:, coefficient_order[order_index]][mask] = (
            flattened_coefficients[
                coefficient_offset:coefficient_offset + coefficient_count
            ]
        )
        coefficient_offset += coefficient_count

    restored_blocks = np.zeros_like(sorted_blocks)
    restored_blocks[sorted_index, :] = modified_sorted_blocks

    modified_dct = restored_blocks.reshape(
        block_rows, block_cols, 8, 8
    ).swapaxes(1, 2).reshape(image_height, image_width)

    jpeg.coef_arrays[0][:] = modified_dct
    output_path = STEGO_IMAGE_DIR / (
        f"{image_name}_{quality_factor}_Stego.jpg"
    )
    jio.write(jpeg, str(output_path))

    return embedded_symbols, bitsum_emd(embedded_group_flags, group_size)


### Embedding

In [3]:
image_name = "Lena_Gray"
quality_factor = 70
test_image_dir = SOURCE_IMAGE_DIR  # Manually select the test-image directory.
# test_image_dir = GRAY_IMAGE_DIR
# test_image_dir = Path(r"C:/path/to/your/jpeg_images")
target_payload_bits = 10000
group_size = 3

embedded_symbols, embedded_bit_count = diff_group_2_200(
    target_payload_bits,
    image_name,
    quality_factor,
    group_size,
    test_image_dir
)


### Extraction

In [4]:
jpeg = jio.read(
    str(STEGO_IMAGE_DIR / f"{image_name}_{quality_factor}_Stego.jpg")
)
quantized_dct = jpeg.coef_arrays[0]
image_height, image_width = quantized_dct.shape
block_rows = image_height // 8
block_cols = image_width // 8

quant_table = jpeg.quant_tables[0]
coefficient_order = np.append(
    0,
    np.argsort(quant_table.flatten("C")[1:], kind="mergesort") + 1,
)

dct_blocks = quantized_dct.reshape(
    block_rows, 8, block_cols, 8
).swapaxes(1, 2).reshape(block_rows * block_cols, 64).astype(int)

last_nonzero_positions = np.zeros(block_rows * block_cols, dtype=int)
for block_index in range(block_rows * block_cols):
    ordered_coefficients = dct_blocks[block_index, coefficient_order]
    nonzero_positions = np.where(ordered_coefficients != 0)[0]

    if len(nonzero_positions) > 0:
        last_nonzero_positions[block_index] = nonzero_positions[-1]

sorted_index = np.argsort(last_nonzero_positions)
sorted_blocks = dct_blocks[sorted_index, :]
sorted_last_nonzero = last_nonzero_positions[sorted_index]

embeddable_coefficients = np.array([], dtype=int)
for order_index in range(1, len(coefficient_order)):
    coefficients = sorted_blocks[:, coefficient_order[order_index]]
    embeddable_coefficients = np.concatenate(
        (
            embeddable_coefficients,
            coefficients[sorted_last_nonzero > order_index],
        )
    )

coefficient_groups = []
for start_index in range(0, len(embeddable_coefficients), group_size):
    coefficient_groups.append(
        embeddable_coefficients[start_index:start_index + group_size]
    )

extracted_symbols = ""
extracted_group_flags = ""

for group in coefficient_groups:
    if (
        len(group) == group_size
        and bitsum_emd(extracted_group_flags, group_size)
        <= target_payload_bits
    ):
        extracted_group_flags += "0"
        extracted_symbols += str(ftake(group, group_size))


In [5]:
if embedded_symbols.strip() == extracted_symbols.strip():
    print("True")
else:
    print("False")


True


### Experiments

In [6]:
image_names = np.array([
    "Lena_Gray",
    "Baboon_Gray",
    "Sailboat_Gray",
    "elaine",
])
quality_factor = 70
image_name = image_names[0]

# Manually select the directory containing the JPEG test images.
test_image_dir = SOURCE_IMAGE_DIR
# test_image_dir = GRAY_IMAGE_DIR
# test_image_dir = Path(r"C:/path/to/your/jpeg_images")


In [7]:
payload_start = 5000
payload_stop = 67000
payload_step = 6000
payload_levels = np.arange(payload_start, payload_stop, payload_step)
embeddable_coefficient_count = found_max(
    image_name,
    quality_factor,
    2,
    1,
    test_image_dir
)

group_size_symbol = sp.symbols("x")
max_group_sizes = np.zeros(len(payload_levels), dtype=int)

valid_count = 0
for payload_bits in payload_levels:
    if payload_bits < (
        embeddable_coefficient_count // 2
    ) * math.log2(2 * 2 + 1):
        solution = sp.nsolve(
            (
                embeddable_coefficient_count / group_size_symbol
            )
            * sp.log(2 * group_size_symbol + 1, 2)
            - payload_bits,
            group_size_symbol,
            2,
        )
        max_group_sizes[valid_count] = np.floor(solution)
        valid_count += 1
    else:
        break

max_group_sizes = max_group_sizes[max_group_sizes != 0]
max_group_sizes


array([88, 32, 18, 11,  8,  6,  5,  4,  3,  2,  2])

In [8]:
proposed_psnr = np.zeros(len(payload_levels))
proposed_ssim = np.zeros(len(payload_levels))

original_image = plt.imread(
    test_image_dir / f"{image_name}_{quality_factor}.jpg"
)

for payload_index, payload_bits in enumerate(
    range(payload_start, payload_stop, payload_step)
):
    best_psnr = 0
    best_ssim = 0

    for group_size in range(2, max_group_sizes[payload_index] + 1):
        _, embedded_capacity_bits = diff_group_2_200(
            payload_bits,
            image_name,
            quality_factor,
            group_size,
            test_image_dir
        )

        if embedded_capacity_bits >= payload_bits:
            stego_image = plt.imread(
                STEGO_IMAGE_DIR
                / f"{image_name}_{quality_factor}_Stego.jpg"
            )

            current_psnr = psnr(original_image, stego_image)

            if current_psnr >= best_psnr:
                best_psnr = np.round(current_psnr, 2)
                best_ssim = np.round(
                    ssim(
                        original_image,
                        stego_image,
                        channel_axis=None,
                        data_range=255,
                    ),
                    5,
                )

    proposed_psnr[payload_index] = best_psnr
    proposed_ssim[payload_index] = best_ssim
    print("Payload Bits: %d" % payload_bits, end=", ")
    print("PSNR= %6.2f" % proposed_psnr[payload_index], end=", ")
    print("SSIM= %6.5f" % proposed_ssim[payload_index])


Payload Bits: 5000, PSNR=  55.20, SSIM= 0.99863
Payload Bits: 11000, PSNR=  50.76, SSIM= 0.99638
Payload Bits: 17000, PSNR=  48.20, SSIM= 0.99329
Payload Bits: 23000, PSNR=  46.40, SSIM= 0.98998
Payload Bits: 29000, PSNR=  44.91, SSIM= 0.98648
Payload Bits: 35000, PSNR=  43.72, SSIM= 0.98179
Payload Bits: 41000, PSNR=  42.64, SSIM= 0.97808
Payload Bits: 47000, PSNR=  41.58, SSIM= 0.97380
Payload Bits: 53000, PSNR=  40.40, SSIM= 0.96871
Payload Bits: 59000, PSNR=  39.15, SSIM= 0.96342
Payload Bits: 65000, PSNR=  36.73, SSIM= 0.95334


In [12]:
proposed_psnr

array([55.2 , 50.76, 48.2 , 46.4 , 44.91, 43.72, 42.64, 41.58, 40.4 ,
       39.15, 36.73])

In [13]:
proposed_ssim

array([0.99863, 0.99638, 0.99329, 0.98998, 0.98648, 0.98179, 0.97808,
       0.9738 , 0.96871, 0.96342, 0.95334])